# Transfer Evaluation: Negation Understanding in DistilBERT

This notebook evaluates trained probes on the JinaAI Negation Dataset to test:
1. **Transfer learning**: How well do SST-2 trained probes generalize to negation pairs?
2. **Distillation hypothesis**: Is negation understanding compressed into Layer 3?

## Prerequisites
Run this notebook **after** completing the sweep notebooks:
- `07_sweep_cls.ipynb`
- `07_sweep_mean.ipynb`
- `07_sweep_token.ipynb`

## Metrics
- **Flip Accuracy**: % of pairs where probe correctly predicts opposite sentiment for negated version
- **Prediction Confidence**: Average confidence in predictions
- **Combined Score**: 0.7 * Flip Accuracy + 0.3 * Confidence


## 1. Setup & Configuration


In [ ]:
#@title **Configure Environment** { display-mode: "form" }
import os
import psutil

# Check resources
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

# RAM check
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Available RAM: {ram_gb:.1f} GB")

if ram_gb > 25:
    print("  High-RAM mode detected")
    BATCH_SIZE = 64
else:
    print("  Standard RAM mode")
    BATCH_SIZE = 32

# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    DEVICE = "cuda"
else:
    print("No GPU detected, using CPU")
    DEVICE = "cpu"
    BATCH_SIZE = 16

print(f"\nConfiguration:")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print("=" * 60)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone or pull the repository
!git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
%cd Negation-Origin-Tracing


In [ ]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm scipy


In [ ]:
# Set paths
import os
from datetime import datetime

DRIVE_PATH = '/content/drive/MyDrive/NOT_results'
OUTPUT_DIR = os.path.join(DRIVE_PATH, 'transfer_evaluation')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Results from sweeps: {DRIVE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Check Sweep Status & Load Probes


In [ ]:
import json
import glob

# Check which sweeps are complete
sweeps = {
    'cls': os.path.join(DRIVE_PATH, 'sweep_cls'),
    'mean': os.path.join(DRIVE_PATH, 'sweep_mean'),
    'token': os.path.join(DRIVE_PATH, 'sweep_token'),
}

print("Sweep Status:")
print("=" * 50)

available_probes = []

for pooling, sweep_dir in sweeps.items():
    results_file = os.path.join(sweep_dir, f'results_{pooling}.json')
    
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            results = json.load(f)
        
        # Find checkpoints
        for result in results:
            layer = result.get('layer_idx', result.get('layer', -1))
            ckpt_path = result.get('checkpoint_path', '')
            
            # Try to find checkpoint in sweep directory
            if not ckpt_path or not os.path.exists(ckpt_path):
                # Look for checkpoint in expected location
                ckpt_pattern = os.path.join(sweep_dir, f'layer_{layer}_pooling_{pooling}', 'checkpoints', 'best*.ckpt')
                ckpt_matches = glob.glob(ckpt_pattern)
                if ckpt_matches:
                    ckpt_path = ckpt_matches[0]
            
            if ckpt_path and os.path.exists(ckpt_path):
                available_probes.append({
                    'layer': layer,
                    'pooling': pooling,
                    'checkpoint': ckpt_path,
                    'val_acc': result.get('test_accuracy', result.get('val_accuracy', 0)),
                })
        
        print(f"  {pooling.upper()}: {len(results)} experiments found")
    else:
        print(f"  {pooling.upper()}: Not found")

print(f"\nTotal probes with checkpoints: {len(available_probes)}")

if len(available_probes) == 0:
    print("\n ERROR: No probe checkpoints found!")
    print("Please run the sweep notebooks first (07_sweep_*.ipynb)")


In [ ]:
# Display available probes
import pandas as pd

if available_probes:
    probes_df = pd.DataFrame(available_probes)
    print("Available Probes:")
    print(probes_df[['layer', 'pooling', 'val_acc']].to_string(index=False))


## 3. Download Negation Dataset


In [ ]:
# Download JinaAI Negation Dataset
NEGATION_DATA_DIR = 'data/raw/negation'

if not os.path.exists(os.path.join(NEGATION_DATA_DIR, 'test.parquet')):
    print("Downloading JinaAI Negation Dataset...")
    !python src/data/download_negation.py --output_dir {NEGATION_DATA_DIR}
else:
    print("Negation dataset already exists!")
    
# Show dataset info
for split in ['train', 'test']:
    path = os.path.join(NEGATION_DATA_DIR, f'{split}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  {split}: {len(df)} examples")


## 4. Load Model and Evaluation Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm
import numpy as np

# Load base model
MODEL_NAME = "distilbert-base-uncased"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)
base_model = base_model.to(DEVICE)
base_model.eval()

print(f"Model loaded on {DEVICE}")
print(f"Hidden size: {base_model.config.hidden_size}")
print(f"Num layers: {base_model.config.num_hidden_layers}")


In [ ]:
# Define probe loading function
import sys
sys.path.insert(0, '.')

from src.models import BaseModule

def load_probe_from_checkpoint(checkpoint_path, layer_idx, pooling_strategy):
    """
    Load a trained probe from checkpoint.
    Returns the linear probe layer.
    """
    # Load the full model from checkpoint
    model = BaseModule.load_from_checkpoint(
        checkpoint_path,
        model_name=MODEL_NAME,
        mode="probe",
        probe_layer=layer_idx,
        pooling_strategy=pooling_strategy,
    )
    model = model.to(DEVICE)
    model.eval()
    
    # Extract just the probe (linear layer)
    probe = model.probes[f"layer_{layer_idx}"]
    
    return model, probe

print("Probe loading function defined.")


In [ ]:
# Load negation dataset
from src.datasets.dataset import NegationTripletDataset
from torch.utils.data import DataLoader

# Use test split for evaluation
test_path = os.path.join(NEGATION_DATA_DIR, 'test.parquet')
if not os.path.exists(test_path):
    # Fall back to train if test doesn't exist
    test_path = os.path.join(NEGATION_DATA_DIR, 'train.parquet')
    print("Using train split (test not found)")

negation_dataset = NegationTripletDataset(
    data_path=test_path,
    tokenizer=tokenizer,
    max_length=128,
)

negation_loader = DataLoader(
    negation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)

print(f"\nLoaded {len(negation_dataset)} negation pairs")


## 5. Evaluation Functions


In [ ]:
def extract_features(model, input_ids, attention_mask, layer_idx, pooling_strategy):
    """
    Extract features from a specific layer using the specified pooling strategy.
    """
    with torch.no_grad():
        outputs = model.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        # Get hidden states for the specified layer
        # hidden_states[0] is embeddings, hidden_states[1] is layer 0, etc.
        hidden_states = outputs.hidden_states[layer_idx + 1]
        
        # Apply pooling
        if pooling_strategy == "cls":
            features = hidden_states[:, 0, :]
        elif pooling_strategy == "mean":
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_hidden = torch.sum(hidden_states * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            features = sum_hidden / sum_mask
        elif pooling_strategy == "token":
            # Pool around "not" token
            not_token_id = tokenizer.convert_tokens_to_ids("not")
            batch_size = hidden_states.size(0)
            pooled = []
            
            for i in range(batch_size):
                not_positions = (input_ids[i] == not_token_id).nonzero(as_tuple=True)[0]
                
                if len(not_positions) > 0:
                    not_idx = not_positions[0].item()
                    start_idx = max(0, not_idx - 1)
                    end_idx = min(hidden_states.size(1), not_idx + 2)
                    valid_mask = attention_mask[i, start_idx:end_idx].bool()
                    if valid_mask.any():
                        pooled_hidden = hidden_states[i, start_idx:end_idx, :][valid_mask].mean(dim=0)
                    else:
                        pooled_hidden = hidden_states[i, not_idx, :]
                else:
                    pooled_hidden = hidden_states[i, 0, :]
                
                pooled.append(pooled_hidden)
            
            features = torch.stack(pooled)
        else:
            raise ValueError(f"Unknown pooling strategy: {pooling_strategy}")
        
        return features


def evaluate_probe_on_negation(model, probe, negation_loader, layer_idx, pooling_strategy):
    """
    Evaluate a probe on the negation dataset.
    
    Returns:
        dict with flip_accuracy, avg_confidence, and other metrics
    """
    model.eval()
    probe.eval()
    
    total_pairs = 0
    correct_flips = 0
    anchor_confidences = []
    negative_confidences = []
    
    with torch.no_grad():
        for batch in tqdm(negation_loader, desc=f"Layer {layer_idx}, {pooling_strategy}", leave=False):
            # Move to device
            anchor_ids = batch['anchor_input_ids'].to(DEVICE)
            anchor_mask = batch['anchor_attention_mask'].to(DEVICE)
            negative_ids = batch['negative_input_ids'].to(DEVICE)
            negative_mask = batch['negative_attention_mask'].to(DEVICE)
            
            # Extract features
            anchor_features = extract_features(
                model, anchor_ids, anchor_mask, layer_idx, pooling_strategy
            )
            negative_features = extract_features(
                model, negative_ids, negative_mask, layer_idx, pooling_strategy
            )
            
            # Get predictions
            anchor_logits = probe(anchor_features)
            negative_logits = probe(negative_features)
            
            # Get predicted classes
            anchor_preds = torch.argmax(anchor_logits, dim=-1)
            negative_preds = torch.argmax(negative_logits, dim=-1)
            
            # Get confidences (probability of predicted class)
            anchor_probs = F.softmax(anchor_logits, dim=-1)
            negative_probs = F.softmax(negative_logits, dim=-1)
            
            anchor_conf = anchor_probs.max(dim=-1).values
            negative_conf = negative_probs.max(dim=-1).values
            
            # Count flips (predictions should differ for anchor vs negative)
            flips = (anchor_preds != negative_preds).sum().item()
            correct_flips += flips
            total_pairs += len(anchor_ids)
            
            anchor_confidences.extend(anchor_conf.cpu().numpy())
            negative_confidences.extend(negative_conf.cpu().numpy())
    
    flip_accuracy = correct_flips / total_pairs if total_pairs > 0 else 0
    avg_anchor_conf = np.mean(anchor_confidences)
    avg_negative_conf = np.mean(negative_confidences)
    avg_confidence = (avg_anchor_conf + avg_negative_conf) / 2
    
    # Combined score (weighted)
    combined_score = 0.7 * flip_accuracy + 0.3 * avg_confidence
    
    return {
        'layer': layer_idx,
        'pooling': pooling_strategy,
        'flip_accuracy': flip_accuracy,
        'avg_confidence': avg_confidence,
        'avg_anchor_confidence': avg_anchor_conf,
        'avg_negative_confidence': avg_negative_conf,
        'combined_score': combined_score,
        'total_pairs': total_pairs,
        'correct_flips': correct_flips,
    }

print("Evaluation functions defined.")


## 6. Run Transfer Evaluation


In [ ]:
import time
from datetime import datetime, timedelta

# Run evaluation on all available probes
print("=" * 60)
print("TRANSFER EVALUATION")
print("=" * 60)
print(f"Evaluating {len(available_probes)} probes on {len(negation_dataset)} negation pairs")
print(f"Started at: {datetime.now().strftime('%H:%M:%S')}")
print("=" * 60)

all_results = []
start_time = time.time()

for i, probe_info in enumerate(available_probes):
    layer = probe_info['layer']
    pooling = probe_info['pooling']
    checkpoint = probe_info['checkpoint']
    
    print(f"\n[{i+1}/{len(available_probes)}] Layer {layer}, {pooling.upper()} pooling")
    
    try:
        # Load probe
        model, probe = load_probe_from_checkpoint(checkpoint, layer, pooling)
        
        # Evaluate
        result = evaluate_probe_on_negation(
            model, probe, negation_loader, layer, pooling
        )
        
        # Add original validation accuracy
        result['val_accuracy'] = probe_info['val_acc']
        result['checkpoint'] = checkpoint
        
        all_results.append(result)
        
        print(f"    Flip Accuracy: {result['flip_accuracy']:.3f}")
        print(f"    Avg Confidence: {result['avg_confidence']:.3f}")
        print(f"    Combined Score: {result['combined_score']:.3f}")
        
        # Estimate remaining time
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (len(available_probes) - i - 1)
        print(f"    ETA: {timedelta(seconds=int(remaining))}")
        
        # Save intermediate results
        with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
            json.dump(all_results, f, indent=2)
        
        # Clear GPU cache
        if DEVICE == "cuda":
            del model, probe
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"    ERROR: {e}")
        all_results.append({
            'layer': layer,
            'pooling': pooling,
            'error': str(e),
        })

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print(f"Total time: {timedelta(seconds=int(time.time() - start_time))}")
print("=" * 60)


In [ ]:
# Display results summary
results_df = pd.DataFrame([r for r in all_results if 'error' not in r])

if len(results_df) > 0:
    print("\nRESULTS SUMMARY")
    print("=" * 60)
    
    # Sort by combined score
    results_df = results_df.sort_values('combined_score', ascending=False)
    
    print(results_df[['layer', 'pooling', 'flip_accuracy', 'avg_confidence', 'combined_score', 'val_accuracy']].to_string(index=False))
    
    # Best configuration
    best = results_df.iloc[0]
    print(f"\nBEST CONFIGURATION:")
    print(f"  Layer {int(best['layer'])}, {best['pooling'].upper()} pooling")
    print(f"  Flip Accuracy: {best['flip_accuracy']:.3f}")
    print(f"  Combined Score: {best['combined_score']:.3f}")


## 7. Hypothesis Testing: Layer 3 Distillation


In [ ]:
from scipy import stats

print("=" * 60)
print("HYPOTHESIS TEST: Layer 3 Distillation")
print("=" * 60)
print("\nHypothesis: Negation understanding is compressed into Layer 3")
print("           (corresponding to BERT layers 7/9)")

if len(results_df) > 0:
    # Get Layer 3 scores vs other layers
    layer_3_scores = results_df[results_df['layer'] == 3]['combined_score'].values
    other_scores = results_df[results_df['layer'] != 3]['combined_score'].values
    
    print(f"\nLayer 3 scores: {layer_3_scores}")
    print(f"Other layers mean: {other_scores.mean():.3f}")
    
    if len(layer_3_scores) > 0 and len(other_scores) > 1:
        # T-test
        t_stat, p_value = stats.ttest_ind(layer_3_scores, other_scores)
        
        # Effect size (Cohen's d)
        pooled_std = np.sqrt(((len(layer_3_scores)-1)*np.var(layer_3_scores) + 
                              (len(other_scores)-1)*np.var(other_scores)) / 
                             (len(layer_3_scores) + len(other_scores) - 2))
        cohens_d = (np.mean(layer_3_scores) - np.mean(other_scores)) / pooled_std if pooled_std > 0 else 0
        
        print(f"\nStatistical Analysis:")
        print(f"  Layer 3 mean: {np.mean(layer_3_scores):.3f}")
        print(f"  Other layers mean: {np.mean(other_scores):.3f}")
        print(f"  t-statistic: {t_stat:.3f}")
        print(f"  p-value: {p_value:.4f}")
        print(f"  Cohen's d: {cohens_d:.3f}")
        
        if p_value < 0.05 and np.mean(layer_3_scores) > np.mean(other_scores):
            print(f"\n  HYPOTHESIS SUPPORTED (p < 0.05)")
            print(f"  Layer 3 shows significantly better negation understanding.")
        elif p_value < 0.05:
            print(f"\n  HYPOTHESIS NOT SUPPORTED")
            print(f"  Layer 3 performs significantly WORSE than other layers.")
        else:
            print(f"\n  NO SIGNIFICANT DIFFERENCE (p >= 0.05)")
            print(f"  Cannot confirm Layer 3 has special negation understanding.")
    
    # Layer-by-layer analysis
    print("\nPerformance by Layer:")
    layer_stats = results_df.groupby('layer').agg({
        'flip_accuracy': 'mean',
        'combined_score': 'mean',
    }).round(3)
    print(layer_stats)
    
    # Best layer
    best_layer = layer_stats['combined_score'].idxmax()
    print(f"\nBest performing layer: {best_layer}")


## 8. Visualizations


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(16, 12))

# BERT layer mapping for DistilBERT
bert_mapping = {0: 0, 1: 2, 2: 4, 3: 7, 4: 9, 5: 11}

if len(results_df) > 0:
    # Plot 1: Flip Accuracy by Layer (lines for each pooling)
    ax1 = plt.subplot(2, 2, 1)
    for pooling in ['cls', 'mean', 'token']:
        pool_data = results_df[results_df['pooling'] == pooling].sort_values('layer')
        if len(pool_data) > 0:
            ax1.plot(pool_data['layer'], pool_data['flip_accuracy'], 
                    marker='o', label=pooling.upper(), linewidth=2, markersize=8)
    
    ax1.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Layer 3 (hypothesis)')
    ax1.set_xlabel('Layer', fontsize=12)
    ax1.set_ylabel('Flip Accuracy', fontsize=12)
    ax1.set_title('Negation Flip Accuracy by Layer', fontsize=14)
    ax1.legend()
    ax1.set_xticks(range(6))
    ax1.set_ylim(0, 1)
    
    # Plot 2: Heatmap of Combined Scores
    ax2 = plt.subplot(2, 2, 2)
    pivot_data = results_df.pivot(index='layer', columns='pooling', values='combined_score')
    sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax2,
                cbar_kws={'label': 'Combined Score'})
    ax2.set_title('Combined Score: Layer x Pooling', fontsize=14)
    ax2.set_xlabel('Pooling Strategy', fontsize=12)
    ax2.set_ylabel('Layer', fontsize=12)
    
    # Plot 3: Layer Comparison Bar Chart
    ax3 = plt.subplot(2, 2, 3)
    layer_means = results_df.groupby('layer')['combined_score'].mean().sort_index()
    colors = ['#3498db'] * 6
    if 3 in layer_means.index:
        colors[3] = '#e74c3c'  # Highlight layer 3
    bars = ax3.bar(layer_means.index, layer_means.values, color=colors, edgecolor='black')
    ax3.set_xlabel('Layer', fontsize=12)
    ax3.set_ylabel('Average Combined Score', fontsize=12)
    ax3.set_title('Average Performance by Layer', fontsize=14)
    ax3.set_xticks(range(6))
    ax3.set_xticklabels([f'{i}\n(BERT {bert_mapping[i]})' for i in range(6)])
    
    # Plot 4: Validation vs Transfer Performance
    ax4 = plt.subplot(2, 2, 4)
    scatter = ax4.scatter(results_df['val_accuracy'], results_df['flip_accuracy'], 
               c=results_df['layer'], cmap='viridis', s=100, edgecolors='black')
    ax4.set_xlabel('Validation Accuracy (SST-2)', fontsize=12)
    ax4.set_ylabel('Flip Accuracy (Transfer)', fontsize=12)
    ax4.set_title('Validation vs Transfer Performance', fontsize=14)
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Layer')
    lims = [0, 1]
    ax4.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
    ax4.set_xlim(lims)
    ax4.set_ylim(lims)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'transfer_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {os.path.join(OUTPUT_DIR, 'transfer_evaluation.png')}")


In [ ]:
# Additional visualization: Distillation Pattern
if len(results_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Expected pattern based on BERT literature
    # Negation peaks around layers 7-9 in BERT-base
    # In DistilBERT (6 layers), this maps to layers 2-4
    expected_pattern = [0.3, 0.4, 0.6, 0.7, 0.5, 0.4]  # Rough expected curve
    
    # Actual pattern
    actual_pattern = results_df.groupby('layer')['combined_score'].mean().sort_index().values
    
    # Pad if not all layers present
    if len(actual_pattern) < 6:
        full_pattern = [0.0] * 6
        for idx, layer in enumerate(results_df.groupby('layer')['combined_score'].mean().sort_index().index):
            full_pattern[int(layer)] = actual_pattern[idx]
        actual_pattern = full_pattern
    
    layers = range(6)
    ax.plot(layers, expected_pattern, 'b--', marker='o', label='Expected (BERT pattern)', linewidth=2)
    ax.plot(layers, actual_pattern, 'r-', marker='s', label='Actual (DistilBERT)', linewidth=2)
    
    # Fill between
    ax.fill_between(layers, expected_pattern, actual_pattern, 
                   where=[a > e for a, e in zip(actual_pattern, expected_pattern)],
                   color='green', alpha=0.2, label='Better than expected')
    ax.fill_between(layers, expected_pattern, actual_pattern,
                   where=[a < e for a, e in zip(actual_pattern, expected_pattern)],
                   color='red', alpha=0.2, label='Worse than expected')
    
    ax.set_xlabel('DistilBERT Layer', fontsize=12)
    ax.set_ylabel('Combined Score', fontsize=12)
    ax.set_title('Distillation Compression: Expected vs Actual', fontsize=14)
    ax.legend()
    ax.set_xticks(range(6))
    ax.set_xticklabels([f'{i}\n(BERT {bert_mapping[i]})' for i in range(6)])
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'distillation_pattern.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to: {os.path.join(OUTPUT_DIR, 'distillation_pattern.png')}")


## 9. Generate Report


In [ ]:
# Generate markdown report
report = f"""# Transfer Evaluation Report

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Overview

- **Dataset**: JinaAI Negation Dataset
- **Test pairs**: {len(negation_dataset)}
- **Probes evaluated**: {len(available_probes)}
- **Model**: DistilBERT (distilbert-base-uncased)

## Results Summary

"""

if len(results_df) > 0:
    # Best configuration
    best = results_df.iloc[0]
    report += f"""### Best Configuration

| Metric | Value |
|--------|-------|
| Layer | {int(best['layer'])} |
| Pooling | {best['pooling'].upper()} |
| Flip Accuracy | {best['flip_accuracy']:.3f} |
| Combined Score | {best['combined_score']:.3f} |

### All Results

| Layer | Pooling | Flip Acc | Confidence | Combined | Val Acc |
|-------|---------|----------|------------|----------|----------|
"""
    
    for _, row in results_df.iterrows():
        report += f"| {int(row['layer'])} | {row['pooling'].upper()} | {row['flip_accuracy']:.3f} | {row['avg_confidence']:.3f} | {row['combined_score']:.3f} | {row['val_accuracy']:.3f} |\n"
    
    # Hypothesis test results
    layer_3_scores = results_df[results_df['layer'] == 3]['combined_score'].values
    other_scores = results_df[results_df['layer'] != 3]['combined_score'].values
    
    if len(layer_3_scores) > 0 and len(other_scores) > 1:
        t_stat, p_value = stats.ttest_ind(layer_3_scores, other_scores)
        
        report += f"""
## Hypothesis Test: Layer 3 Distillation

**Hypothesis**: Negation understanding is compressed into Layer 3 in DistilBERT
(corresponding to BERT layers 7/9).

| Metric | Value |
|--------|-------|
| Layer 3 mean score | {np.mean(layer_3_scores):.3f} |
| Other layers mean | {np.mean(other_scores):.3f} |
| t-statistic | {t_stat:.3f} |
| p-value | {p_value:.4f} |

**Conclusion**: """
        
        if p_value < 0.05 and np.mean(layer_3_scores) > np.mean(other_scores):
            report += "Hypothesis SUPPORTED. Layer 3 shows significantly better negation understanding.\n"
        elif p_value < 0.05:
            report += "Hypothesis NOT SUPPORTED. Layer 3 performs significantly worse than other layers.\n"
        else:
            report += "NO SIGNIFICANT DIFFERENCE. Cannot confirm Layer 3 has special negation understanding.\n"

report += f"""
## Visualizations

- `transfer_evaluation.png`: Main evaluation plots
- `distillation_pattern.png`: Expected vs actual distillation pattern

## Files

- `results.json`: Raw evaluation results
- `TRANSFER_REPORT.md`: This report
"""

# Save report
report_path = os.path.join(OUTPUT_DIR, 'TRANSFER_REPORT.md')
with open(report_path, 'w') as f:
    f.write(report)

print(f"Report saved to: {report_path}")
print("\n" + "=" * 60)
print(report)


In [ ]:
# Save final results
with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nAll results saved to: {OUTPUT_DIR}")
print("\nFiles:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  - {f}")


## 10. Summary

This notebook evaluated trained probes on the JinaAI Negation Dataset to test transfer learning and the distillation hypothesis. Check the TRANSFER_REPORT.md file in the output directory for detailed results.
